In [9]:
import os
import math
import time
import json
import hashlib
from pathlib import Path
from glob import glob

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc

import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader
from tqdm import tqdm

# --- Paths (your provided layout) ---
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH = DATASET_ROOT / "hf" / "merrec"
print("DATASET_PATH =", DATASET_PATH)
assert DATASET_PATH.exists(), f"Path not found: {DATASET_PATH}"

# --- Device (Mac M2) ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

# For stability on some macOS setups (optional)
# torch.set_num_threads(max(1, os.cpu_count() // 2))


DATASET_PATH = /Volumes/T5 EVO/hf/merrec
Device: cpu


In [21]:
CFG = {
    # Hash bucket sizes (tradeoff: memory vs collisions)
    # If you go too small → collisions hurt quality.
    # If you go too big → embedding memory blows up.
    "user_buckets": 2_000_000,   # ~2M
    "item_buckets": 2_000_000,   # ~2M

    # Model hyperparams (paper used embed_dim=32, hidden=128; keep that)
    "embed_dim": 32,
    "hidden": 128,

    # Training
    "lr": 5e-5,
    "epochs": 1,                 # full dataset epoch is huge; start with 1
    "batch_size": 4096,          # reduce if MPS OOM (try 1024/2048)
    "num_workers": 0,            # keep 0 on macOS for stability
    "prefetch_batches": 2,

    # Data streaming
    "shuffle_files": True,       # shuffle file order each epoch
    "max_rows_per_file": None,   # optionally cap per-file rows for debugging
    
    "max_total_rows": 10_000_000,  # Optionally limit total rows for quick tests (set to None to disable)
    # Checkpointing
    "ckpt_dir": str(Path.cwd() / "checkpoints_nfm_hash"),
    "ckpt_every_steps": 2000,    # save every N steps
    "resume": True,              # resume if checkpoint exists

    # Columns (must exist in your Parquet schema)
    "col_user": "user_id",
    "col_item": "product_id",
    "col_action": "event_id",
    "positive_action": "item_view",
}

Path(CFG["ckpt_dir"]).mkdir(parents=True, exist_ok=True)
CFG


{'user_buckets': 2000000,
 'item_buckets': 2000000,
 'embed_dim': 32,
 'hidden': 128,
 'lr': 5e-05,
 'epochs': 1,
 'batch_size': 4096,
 'num_workers': 0,
 'prefetch_batches': 2,
 'shuffle_files': True,
 'max_rows_per_file': None,
 'max_total_rows': 10000000,
 'ckpt_dir': '/Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/notebooks/checkpoints_nfm_hash',
 'ckpt_every_steps': 2000,
 'resume': True,
 'col_user': 'user_id',
 'col_item': 'product_id',
 'col_action': 'event_id',
 'positive_action': 'item_view'}

In [3]:
def stable_hash_to_bucket(x: str, buckets: int) -> int:
    """
    Stable hashing across runs/machines (unlike Python's built-in hash()).
    """
    if x is None:
        x = ""
    if not isinstance(x, str):
        x = str(x)
    h = hashlib.blake2b(x.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(h, "little") % buckets

In [11]:
class MerRecStreamingCTR(IterableDataset):
    """
    Streams Parquet files and yields batches.
    Now supports limiting training to a subset of rows.
    """

    def __init__(self, dataset_path: Path, cfg: dict, epoch: int = 0):
        super().__init__()
        self.dataset_path = Path(dataset_path)
        self.cfg = cfg
        self.epoch = epoch

        # discover parquet files
        self.files = sorted(glob(str(self.dataset_path / "**" / "*.parquet"), recursive=True))

        if not self.files:
            raise FileNotFoundError(f"No parquet files found under {self.dataset_path}")

    def _iter_files(self):
        files = self.files[:]

        if self.cfg["shuffle_files"]:
            g = torch.Generator()
            g.manual_seed(12345 + self.epoch)
            idx = torch.randperm(len(files), generator=g).tolist()
            files = [files[i] for i in idx]

        return files

    def __iter__(self):

        bs = self.cfg["batch_size"]
        ub = self.cfg["user_buckets"]
        ib = self.cfg["item_buckets"]

        col_u = self.cfg["col_user"]
        col_i = self.cfg["col_item"]
        col_a = self.cfg["col_action"]
        pos_action = self.cfg["positive_action"]

        max_rows_per_file = self.cfg["max_rows_per_file"]
        max_total_rows = self.cfg.get("max_total_rows", None)

        batch_u, batch_i, batch_y = [], [], []

        total_rows = 0

        for fp in self._iter_files():

            try:
                dataset = ds.dataset(fp, format="parquet")

                scanner = dataset.scanner(
                    columns=[col_u, col_i, col_a],
                    batch_size=65536
                )

                seen_rows = 0

                for record_batch in scanner.to_batches():

                    if max_rows_per_file is not None and seen_rows >= max_rows_per_file:
                        break

                    users = record_batch.column(col_u).to_pylist()
                    items = record_batch.column(col_i).to_pylist()
                    acts = record_batch.column(col_a).to_pylist()

                    for u, it, act in zip(users, items, acts):

                        # GLOBAL LIMIT FOR SUBSET TRAINING
                        if max_total_rows is not None and total_rows >= max_total_rows:
                            return

                        total_rows += 1

                        if u is None or it is None:
                            continue

                        y = 1.0 if act == pos_action else 0.0

                        u_id = stable_hash_to_bucket(u, ub)
                        i_id = stable_hash_to_bucket(it, ib)

                        batch_u.append(u_id)
                        batch_i.append(i_id)
                        batch_y.append(y)

                        if len(batch_y) >= bs:
                            yield (
                                torch.tensor(batch_u, dtype=torch.long),
                                torch.tensor(batch_i, dtype=torch.long),
                                torch.tensor(batch_y, dtype=torch.float32),
                            )

                            batch_u, batch_i, batch_y = [], [], []

                    seen_rows += len(users)

            except Exception as e:
                print(f"[WARN] Failed reading {fp}: {e}")
                continue

        if batch_y:
            yield (
                torch.tensor(batch_u, dtype=torch.long),
                torch.tensor(batch_i, dtype=torch.long),
                torch.tensor(batch_y, dtype=torch.float32),
            )

In [12]:
class NFMHashed(nn.Module):
    """
    Minimal NFM for two categorical fields (user, item):
    - embeddings for user + item
    - Bi-Interaction pooling (FM-style) over the two embeddings
    - MLP to output sigmoid probability

    Also includes linear terms (common in FM variants) for stability.
    """
    def __init__(self, user_buckets: int, item_buckets: int, embed_dim: int = 32, hidden: int = 128):
        super().__init__()
        self.user_emb = nn.Embedding(user_buckets, embed_dim)
        self.item_emb = nn.Embedding(item_buckets, embed_dim)

        # Optional linear terms (wide part)
        self.user_lin = nn.Embedding(user_buckets, 1)
        self.item_lin = nn.Embedding(item_buckets, 1)
        self.bias = nn.Parameter(torch.zeros(1))

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

        # Init
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_lin.weight)
        nn.init.zeros_(self.item_lin.weight)

    @staticmethod
    def bi_interaction(u, v):
        # For 2 fields, this reduces to elementwise product:
        # 0.5 * ((u+v)^2 - (u^2+v^2)) = u*v
        return u * v

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx)
        v = self.item_emb(item_idx)

        z = self.bi_interaction(u, v)  # [B, D]
        deep = self.mlp(z).squeeze(-1)

        wide = self.user_lin(user_idx).squeeze(-1) + self.item_lin(item_idx).squeeze(-1) + self.bias
        logits = deep + wide
        return torch.sigmoid(logits)


In [13]:
def ckpt_path(ckpt_dir: str) -> Path:
    return Path(ckpt_dir) / "latest.pt"

def save_ckpt(path: Path, model, opt, epoch: int, step: int, running_loss: float, cfg: dict):
    tmp = path.with_suffix(".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "opt": opt.state_dict(),
            "epoch": epoch,
            "step": step,
            "running_loss": running_loss,
            "cfg": cfg,
            "time": time.time(),
        },
        tmp,
    )
    tmp.replace(path)

def load_ckpt(path: Path, model, opt):
    obj = torch.load(path, map_location="cpu")
    model.load_state_dict(obj["model"])
    opt.load_state_dict(obj["opt"])
    return obj


In [15]:
print(CFG.get("max_total_rows"))

20000000


In [24]:
model = NFMHashed(
    user_buckets=CFG["user_buckets"],
    item_buckets=CFG["item_buckets"],
    embed_dim=CFG["embed_dim"],
    hidden=CFG["hidden"],
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
loss_fn = nn.BCELoss()

start_epoch = 0
global_step = 0
running_loss = 0.0

ckpt = ckpt_path(CFG["ckpt_dir"])
if CFG["resume"] and ckpt.exists():
    obj = load_ckpt(ckpt, model, opt)
    start_epoch = int(obj.get("epoch", 0))
    global_step = int(obj.get("step", 0))
    running_loss = float(obj.get("running_loss", 0.0))
    print(f"Resumed from {ckpt} @ epoch={start_epoch} step={global_step}")

model.train()

for epoch in range(start_epoch, CFG["epochs"]):
    ds_iter = MerRecStreamingCTR(DATASET_PATH, CFG, epoch=epoch)
    loader = DataLoader(
        ds_iter,
        batch_size=None,                 # dataset already yields batches
        num_workers=CFG["num_workers"],
        prefetch_factor=CFG["prefetch_batches"] if CFG["num_workers"] > 0 else None,
        pin_memory=False,
    )

    pbar = tqdm(loader, desc=f"epoch {epoch}", mininterval=1.0)
    epoch_loss_sum = 0.0
    epoch_batches = 0
    t0 = time.time()

    for user_idx, item_idx, y in pbar:
        user_idx = user_idx.to(device)
        item_idx = item_idx.to(device)
        y = y.to(device)

        opt.zero_grad(set_to_none=True)
        pred = model(user_idx, item_idx)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        l = float(loss.item())
        running_loss = 0.98 * running_loss + 0.02 * l if global_step > 0 else l
        epoch_loss_sum += l
        epoch_batches += 1
        global_step += 1

        if global_step % 50 == 0:
            pbar.set_postfix(loss=l, ema=running_loss, step=global_step)

        if global_step % CFG["ckpt_every_steps"] == 0:
            save_ckpt(ckpt, model, opt, epoch=epoch, step=global_step, running_loss=running_loss, cfg=CFG)
            pbar.set_postfix(loss=l, ema=running_loss, step=global_step, ckpt="saved")

    dt = time.time() - t0
    avg_epoch_loss = epoch_loss_sum / max(1, epoch_batches)
    print(f"Epoch {epoch} done: avg_loss={avg_epoch_loss:.6f} batches={epoch_batches} time={dt/60:.1f} min")

    # Save end-of-epoch checkpoint
    save_ckpt(ckpt, model, opt, epoch=epoch+1, step=global_step, running_loss=running_loss, cfg=CFG)
    print(f"Saved checkpoint: {ckpt}")


epoch 0: 2441it [3:32:45,  5.23s/it, ema=0.459, loss=0.525, step=2400]            


Epoch 0 done: avg_loss=0.545387 batches=2441 time=212.8 min
Saved checkpoint: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/notebooks/checkpoints_nfm_hash/latest.pt


In [26]:
final_path = Path(CFG["ckpt_dir"]) / "nfm_hashed_final.pt"
torch.save({"model": model.state_dict(), "cfg": CFG}, final_path)
print("Saved:", final_path)


Saved: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/notebooks/checkpoints_nfm_hash/nfm_hashed_finalik.pt
